In [ ]:
# TensorFlow와 Keras의 기능을 사용하기 위한 모듈 로드
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential              # 순차적 모델 모듈
from tensorflow.keras.layers import Dense, Input, Dropout   # 밀집층 관련 모듈
from tensorflow.keras.callbacks import EarlyStopping        # 과적합 방지

from sklearn.model_selection import train_test_split        

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

In [ ]:
# 데이터 로드
df = pd.read_csv('../AI_PBL_data/customer_data_balanced.csv')
df

,Age,Tenure,MonthlySpending_KRW,ContractType,CustomerServiceCalls,IsChurn
0,56,29,249010,1,6,0
1,69,44,54542,1,6,0
2,46,53,30651,1,1,0
3,32,24,119239,0,5,1
4,60,58,361075,1,1,0
...,...,...,...,...,...,...
1995,63,30,249204,1,4,0
1996,67,33,116393,1,6,0
1997,69,36,409458,0,3,0
1998,24,18,96312,2,4,0


In [ ]:
# 데이터 보존을 위한 copy
data = df.copy()

# 원 핫 인코딩
data = pd.get_dummies(data,columns=['ContractType']).astype(int)

# X, y 데이터 분리
X = data.drop(columns='IsChurn')
y = data['IsChurn']

In [ ]:
# 테이터 분리 7: 1.5 : 1.5
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

In [ ]:
# 데이터 표준화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# 모델 생성
model = Sequential([
    Input(shape=(X_train.shape[1], )),  # 입력층
    Dense(8, activation='relu'),

    # 과적합 방지
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

In [ ]:
# 모델 컴파일
model.compile(
    optimizer='adam',
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)

In [35]:
# 과적합 방지
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
# 모델 훈련 (과적합 방지)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[early_stopping]
)

Epoch 1/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4600 - loss: 0.8203 - val_accuracy: 0.4233 - val_loss: 0.7801
Epoch 2/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4864 - loss: 0.7758 - val_accuracy: 0.4800 - val_loss: 0.7414
Epoch 3/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5321 - loss: 0.7329 - val_accuracy: 0.5100 - val_loss: 0.7149
Epoch 4/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5393 - loss: 0.7250 - val_accuracy: 0.5367 - val_loss: 0.6958
Epoch 5/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5650 - loss: 0.7056 - val_accuracy: 0.5533 - val_loss: 0.6813
Epoch 6/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5721 - loss: 0.6891 - val_accuracy: 0.5933 - val_loss: 0.6688
Epoch 7/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5700 - loss: 0.6897 - val_accuracy: 0.6033 - val_loss: 0.6604
Epoch 8/50
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5907 - loss: 0.6804 - val_accuracy: 0.6267 - val_loss:

In [ ]:
# 모델 평가
test_loss, test_acc = model.evaluate(X_test, y_test)
print(test_loss)
print(test_acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7200 - loss: 0.5889 
0.5889313220977783
0.7200000286102295


In [ ]:
# 테스트 데이터를 통한 예측 (threshold = 0.5)
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [ ]:
# 평가 지표 출력
accuracy = accuracy_score(y_test, y_pred)   
f1 = f1_score(y_test, y_pred)               

print('정확도(Accuracy):', accuracy)
print('F1-Score:', f1)

print('='*10, '혼동 행렬','='*10)
print(confusion_matrix(y_test, y_pred))

print('='*10, '분류 보고서','='*10)
print(classification_report(y_test, y_pred))

정확도(Accuracy): 0.72
F1-Score: 0.5
========== 혼동 행렬 ==========
[[174  19]
 [ 65  42]]
========== 분류 보고서 ==========
              precision    recall  f1-score   support

           0       0.73      0.90      0.81       193
           1       0.69      0.39      0.50       107

    accuracy                           0.72       300
   macro avg       0.71      0.65      0.65       300
weighted avg       0.71      0.72      0.70       300



보고서

모델 평가 결과, 테스트 데이터 기준으로 약 0.73의 정확도를 보였으나 F1-score는 약 0.5로 나타났다. 혼동 행렬 분석 결과, 비이탈 고객은 잘 분류하는 반면 이탈 고객을 충분히 탐지하지 못하는 문제가 확인된다. 특히 이탈 클래스의 재현율이 낮아 실제 이탈 고객을 상당수 놓치는 경향이 있었다.

이탈 고객을 민감하게 탐지하는 것이 비이탈 고객을 분류하는 것보다 중요한 상황이기 때문에, accuracy가 다소 낮아지더라도 recall 을 높히는 방향이 중요하다. 따라서 threshold 값을 조정해보면서 recall 이 높아지는 방향을 찾아보았다.

In [45]:
# f1_score가 가장 높게 나오는 (이탈율에 가장 민감한) 임계값 찾기
for t in [0.3, 0.4, 0.5, 0.6]: 
    y_pred = (model.predict(X_test) > t).astype(int) 
    
    print(t, accuracy_score(y_test, y_pred)) 
    print(t, f1_score(y_test, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
0.3 0.51
0.3 0.5531914893617021
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
0.4 0.6466666666666666
0.4 0.49038461538461536
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
0.5 0.72
0.5 0.5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
0.6 0.72
0.6 0.4


threshold 값이 0.3일 떄, f-1 스코어가 가장 높게 나타나는 것을 확인하였다.
이후 threshold 값 0.3 으로 평가 지표를 다시 계산하였다.

In [46]:
# threshold 0.3으로 적용 후 평가 지표 출력

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.3).astype(int)

accuracy = accuracy_score(y_test, y_pred) 
f1 = f1_score(y_test, y_pred)             

print('정확도(Accuracy):', accuracy)
print('F1-Score:', f1)

print('='*10, '혼동 행렬','='*10)
print(confusion_matrix(y_test, y_pred))

print('='*10, '분류 보고서','='*10)
print(classification_report(y_test, y_pred))

 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
정확도(Accuracy): 0.51
F1-Score: 0.5531914893617021
========== 혼동 행렬 ==========
[[ 62 131]
 [ 16  91]]
========== 분류 보고서 ==========
              precision    recall  f1-score   support

           0       0.79      0.32      0.46       193
           1       0.41      0.85      0.55       107

    accuracy                           0.51       300
   macro avg       0.60      0.59      0.51       300
weighted avg       0.66      0.51      0.49       300



임계값을 0.3으로 두고 평가해본 결과, 1(이탈 고객) 의 recall 값이 0.85로 나타났다. 혼동 행렬을 볼 때도 비이탈 고객에 대한 분류 정확도는 낮아졌으나 이탈 고객에 대한 정확도는 올라간 것으로 보인다.

데이터에 따라서 초점을 맞추기에 따라 다르겠지만, 현 데이터의 경우 이탈에 민감한 결과를 나타내는 것이 중요하기 때문에 임계값 0.3으로 이탈 고객에 대한 recall을 높히는게 더 적합하다.